# Raw data processing

In [ ]:
# # postprocess_all.py

# import os
# import glob
# import pandas as pd
# import numpy as np

# def parse_volume(x):
#     """'1.61K', '71.47M', '2.3B' 등 단위를 숫자로 변환"""
#     s = str(x).strip()
#     if s.endswith('M'):
#         return float(s[:-1]) * 1_000_000
#     if s.endswith('K'):
#         return float(s[:-1]) * 1_000
#     if s.endswith('B'):
#         return float(s[:-1]) * 1_000_000_000
#     try:
#         return float(s.replace(',', ''))
#     except:
#         return np.nan

# # 1) 원본 Back Test 루트 폴더 & 결과를 저장할 Processed Data 폴더
# # BACK_TEST_ROOT   = r"C:\Users\LabPC\OneDrive\주식\Back Test"
# # PROCESSED_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Processed Data"

# BACK_TEST_ROOT   = r"D:\주식\Back Test"
# PROCESSED_FOLDER = r"D:\주식\Processed Data"

# os.makedirs(PROCESSED_FOLDER, exist_ok=True)

# # 2) 각 종목(하위 폴더)마다 CSV 파일 찾아서 처리
# for company in os.listdir(BACK_TEST_ROOT):
#     company_path = os.path.join(BACK_TEST_ROOT, company)
#     if not os.path.isdir(company_path):
#         continue

#     csv_files = glob.glob(os.path.join(company_path, "*.csv"))
#     for file_path in csv_files:
#         # 3) CSV 불러와서 컬럼명 한글로 변경
#         df = (
#             pd.read_csv(file_path)
#               .rename(columns={
#                   'Date':     '날짜',
#                   'Price':    '종가',
#                   'Open':     '시가',
#                   'High':     '고가',
#                   'Low':      '저가',
#                   'Vol.':     '거래량',
#                   'Change %': '변동 %'
#               })
#         )

#         # 4) 날짜 파싱 & 정렬
#         df['날짜'] = pd.to_datetime(df['날짜'], format='%m/%d/%Y')
#         df = df.sort_values('날짜').reset_index(drop=True)

#         # 4.1) 가격 컬럼(종가, 시가, 고가, 저가) 콤마 제거 후 float 변환
#         for col in ['종가', '시가', '고가', '저가']:
#             df[col] = (
#                 df[col]
#                   .astype(str)
#                   .str.replace(',', '', regex=False)
#                   .astype(float)
#             )

#         # ───────────────────────────────────────────────
#         #  추가: 데이터 시작·종료일 추출
#         # ───────────────────────────────────────────────
#         start_date = df['날짜'].min().strftime('%Y-%m-%d')
#         end_date   = df['날짜'].max().strftime('%Y-%m-%d')
#         df['시작일'] = start_date
#         df['종료일'] = end_date

#         # 5) 거래량·변동 % 숫자 처리
#         df['거래량'] = df['거래량'].apply(parse_volume)
#         df['변동 %'] = (
#             df['변동 %']
#               .astype(str)
#               .str.replace(',', '', regex=False)       # ← 콤마 제거 추가
#               .str.rstrip('%')
#               .astype(float)
#         )

#         # === 지표 계산 ===

#         # ▶ RSI (14일)
#         delta     = df['종가'].diff()
#         gain      = delta.clip(lower=0)
#         loss      = -delta.clip(upper=0)
#         avg_gain  = gain.rolling(window=14).mean()
#         avg_loss  = loss.rolling(window=14).mean()
#         df['RSI (14일)'] = 100 - (100 / (1 + avg_gain/avg_loss))

#         # ▶ Bollinger Bands (20일)
#         m = df['종가'].rolling(window=20).mean()
#         s = df['종가'].rolling(window=20).std()
#         df['볼린저밴드 상단'] = m + 2 * s
#         df['볼린저밴드 하단'] = m - 2 * s

#         # ▶ MACD & Signal
#         ema12 = df['종가'].ewm(span=12, adjust=False).mean()
#         ema26 = df['종가'].ewm(span=26, adjust=False).mean()
#         df['MACD']        = ema12 - ema26
#         df['MACD 시그널'] = df['MACD'].ewm(span=9, adjust=False).mean()

#         # ▶ SMA (5,10,20,60,120,200일)
#         for p in [5, 10, 20, 60, 120, 200]:
#             df[f"SMA {p}일"] = df['종가'].rolling(window=p).mean()

#         # ▶ 가격·거래량 % 변화 (2주,3개월,6개월,1년)
#         periods = {'2주': 10, '3개월': 63, '6개월': 126, '1년': 252}
#         for label, span in periods.items():
#             df[f'가격 상승률 ({label})']  = df['종가'].pct_change(span)  * 100
#             df[f'거래량 상승률 ({label})'] = df['거래량'].pct_change(span) * 100

#         # 6) 저장 (회사명_원본이름_지표포함.csv)
#         base      = os.path.splitext(os.path.basename(file_path))[0]
#         save_name = f"{company}_{base}_지표포함.csv"
#         save_path = os.path.join(PROCESSED_FOLDER, save_name)
#         df.to_csv(save_path, index=False, encoding='utf-8-sig')
#         print(f"✅ Processed: {company} → {save_name} (기간: {start_date} ~ {end_date})")


In [ ]:
import os
import glob
import pandas as pd
import numpy as np

def parse_volume(x):
    """'1.61K', '71.47M', '2.3B' 등 단위를 숫자로 변환"""
    s = str(x).strip()
    if s.endswith('M'):
        return float(s[:-1]) * 1_000_000
    if s.endswith('K'):
        return float(s[:-1]) * 1_000
    if s.endswith('B'):
        return float(s[:-1]) * 1_000_000_000
    try:
        return float(s.replace(',', ''))
    except:
        return np.nan

# 1) 원본 Back Test 루트 폴더 & 결과를 저장할 Processed Data 폴더
# C:\Users\LabPC\OneDrive\
BACK_TEST_ROOT   = r"C:\Users\LabPC\OneDrive\주식\Back Test"
PROCESSED_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Processed Data"

os.makedirs(PROCESSED_FOLDER, exist_ok=True)

# 2) 각 종목(하위 폴더)마다 CSV 파일 찾아서 처리
for company in os.listdir(BACK_TEST_ROOT):
    company_path = os.path.join(BACK_TEST_ROOT, company)
    if not os.path.isdir(company_path):
        continue

    csv_files = glob.glob(os.path.join(company_path, "*.csv"))
    for file_path in csv_files:
        # 3) CSV 불러와서 컬럼명 한글로 변경
        df = (
            pd.read_csv(file_path)
              .rename(columns={
                  'Date':     '날짜',
                  'Price':    '종가',
                  'Open':     '시가',
                  'High':     '고가',
                  'Low':      '저가',
                  'Vol.':     '거래량',
                  'Change %': '변동 %'
              })
        )

        # 4) 날짜 파싱 & 정렬
        df['날짜'] = pd.to_datetime(df['날짜'], format='%m/%d/%Y')
        df = df.sort_values('날짜').reset_index(drop=True)

        # 4.1) 가격 컬럼(종가, 시가, 고가, 저가) 콤마 제거 후 float 변환
        for col in ['종가', '시가', '고가', '저가']:
            df[col] = (
                df[col]
                  .astype(str)
                  .str.replace(',', '', regex=False)
                  .astype(float)
            )

        # ───────────────────────────────────────────────
        #  추가: 데이터 시작·종료일 추출
        # ───────────────────────────────────────────────
        start_date = df['날짜'].min().strftime('%Y-%m-%d')
        end_date   = df['날짜'].max().strftime('%Y-%m-%d')
        df['시작일'] = start_date
        df['종료일'] = end_date

        # 5) 거래량·변동 % 숫자 처리
        df['거래량'] = df['거래량'].apply(parse_volume)
        df['변동 %'] = (
            df['변동 %']
              .astype(str)
              .str.replace(',', '', regex=False)
              .str.rstrip('%')
              .astype(float)
        )

        # === 지표 계산 ===

        # ▶ RSI (14일)
        delta     = df['종가'].diff()
        gain      = delta.clip(lower=0)
        loss      = -delta.clip(upper=0)
        avg_gain  = gain.rolling(window=14).mean()
        avg_loss  = loss.rolling(window=14).mean()
        df['RSI (14일)'] = 100 - (100 / (1 + avg_gain/avg_loss))

        # ▶ Bollinger Bands (20일)
        m = df['종가'].rolling(window=20).mean()
        s = df['종가'].rolling(window=20).std()
        df['볼린저밴드 상단'] = m + 2 * s
        df['볼린저밴드 하단'] = m - 2 * s

        # ▶ MACD & Signal
        ema12 = df['종가'].ewm(span=12, adjust=False).mean()
        ema26 = df['종가'].ewm(span=26, adjust=False).mean()
        df['MACD']        = ema12 - ema26
        df['MACD 시그널'] = df['MACD'].ewm(span=9, adjust=False).mean()

        # ▶ SMA (5,10,20,60,120,200일)
        for p in [5, 10, 20, 60, 120, 200]:
            df[f"SMA {p}일"] = df['종가'].rolling(window=p).mean()

        # ▶ 가격·거래량 % 변화 (2주,3개월,6개월,1년)
        periods = {'2주': 10, '3개월': 63, '6개월': 126, '1년': 252}
        for label, span in periods.items():
            df[f'가격 상승률 ({label})']   = df['종가'].pct_change(span)  * 100
            df[f'거래량 상승률 ({label})'] = df['거래량'].pct_change(span) * 100

        # 6) 저장 (회사명_원본이름_지표포함.csv)
        base      = os.path.splitext(os.path.basename(file_path))[0]
        save_name = f"{company}_{base}_지표포함.csv"
        save_path = os.path.join(PROCESSED_FOLDER, save_name)

        # ───────────────────────────────────────────────
        # ★ 중복 제거: 같은 회사명_원본이름_지표포함.csv 패턴의 이전 파일 삭제
        # ───────────────────────────────────────────────
        pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
        for old_file in glob.glob(pattern):
            try:
                os.remove(old_file)
            except OSError:
                pass  # 삭제 실패해도 무시

        # 새로운 파일 저장
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        print(f"✅ Processed: {company} → {save_name} (기간: {start_date} ~ {end_date})")


# 코드1. 주식을 위한 optimization (VIX 매수, 매도는 Forced buying and selling) 
# 기간부터 설정해

#is_macro_buy/sell 함수로 매크로(VIX) 신호

#is_rule_buy/sell 함수로 기술적 신호

#백테스트 루프에서 매크로 우선 

#비보유 시 매크로 매수 → 보유 시 매크로 매도 → 기술적 신호 적용 순서

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER      = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
MACRO_FOLDER          = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
PARAMETERS_FOLDER     = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH       = os.path.join(PARAMETERS_FOLDER, "parameters.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록을 자동으로 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip()
if sel.lower() == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간을 사용자 입력으로 받기
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일을 YYYY-MM-DD 형식으로 입력하세요 (기본 {default_start}): ").strip()
if start_in:
    START_DATE = start_in
else:
    START_DATE = default_start

end_in = input(f"백테스트 종료일을 YYYY-MM-DD 형식으로 입력하세요 (기본 {default_end}): ").strip()
if end_in:
    END_DATE = end_in
else:
    END_DATE = default_end

print(f"\n▶ 테스트 기간: {START_DATE} 부터 {END_DATE} 까지\n")

new_records = []


# ─────────────────────────────────────────────────────────────
# 2) 선택된 종목 각각에 대해 최적화 수행
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    file_pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    matches = glob.glob(file_pattern)
    if not matches:
        print(f"⚠️ {company}용 파일이 없습니다: {file_pattern}")
        continue
    file_path = matches[0]
    print(f"\n🔍 Optimizing {company} (파일: {os.path.basename(file_path)})")

    # 데이터 로드 & 필터
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)

    # VIX 병합
    vix_pattern = os.path.join(MACRO_FOLDER, "*VIX*.csv")
    vix_files = glob.glob(vix_pattern)
    if not vix_files:
        print(f"⚠️ 경고: '{MACRO_FOLDER}' 폴더에 VIX.csv 파일이 없습니다. VIX 컬럼을 NaN으로 채웁니다.")
        df['VIX'] = np.nan
    else:
        vix_path = vix_files[0]
        vix = pd.read_csv(
            vix_path,
            parse_dates=[0],
            encoding='utf-8-sig',
            names=['날짜','VIX'],
            header=0
        )
        df = df.merge(vix, on='날짜', how='left')

    # 신호 함수 정의
    def is_macro_buy(r,p):  return not np.isnan(r['VIX']) and r['VIX'] >= p['vix_buy_th']
    def is_macro_sell(r,p): return not np.isnan(r['VIX']) and r['VIX'] <= p['vix_sell_th']
    def is_rule_buy(r,p):
        return (
            r['RSI (14일)']           < p['rsi_buy_th'] and
            r['종가']                 < r['볼린저밴드 하단']*(1+p['boll_buffer']) and
            r['MACD']                 > r['MACD 시그널'] and
            r['SMA 5일']              > r['SMA 10일'] and
            r['SMA 5일']              < r['SMA 60일'] and
            r['가격 상승률 (2주)']     < p['tw_price_th'] and
            r['가격 상승률 (3개월)']   < p['tm_price_th'] and
            r['거래량 상승률 (2주)']   < p['tw_vol_th'] and
            r['거래량 상승률 (3개월)'] < p['tm_vol_th']
        )
    def is_rule_sell(r,p): return r['RSI (14일)'] > p['rsi_sell_th']

    # ROI 계산 함수
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_macro_buy(r, p):
                shares, cash = cash/price, 0.0; continue
            if shares > 0 and is_macro_sell(r, p):
                cash, shares = shares*price, 0.0; continue
            if shares == 0 and is_rule_buy(r, p):
                shares, cash = cash/price, 0.0
            elif shares > 0 and is_rule_sell(r, p):
                cash, shares = shares*price, 0.0
        final = cash + shares * df.iloc[-1]['종가']
        return (final - 10_000.0)/10_000.0*100

    # 1단계 TPE 최적화
    def obj_tpe(trial):
        vb = trial.suggest_float("vix_buy_th", 0,100)
        vs = trial.suggest_float("vix_sell_th",0,vb)
        return backtest_roi({
            'vix_buy_th':vb,'vix_sell_th':vs,
            'rsi_buy_th':trial.suggest_float("rsi_buy_th",0,100),
            'boll_buffer':trial.suggest_float("boll_buffer",0,0.1),
            'tw_price_th':trial.suggest_float("tw_price_th",0,20),
            'tm_price_th':trial.suggest_float("tm_price_th",0,50),
            'tw_vol_th':trial.suggest_float("tw_vol_th",0,100),
            'tm_vol_th':trial.suggest_float("tm_vol_th",0,300),
            'rsi_sell_th':trial.suggest_float("rsi_sell_th",0,100),
        })
    tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    tpe.optimize(obj_tpe, n_trials=1500)
    best_tpe = tpe.best_params

    # 파라미터 중요도
    imp = get_param_importances(tpe)
    thr = 0.05
    imp_k = [k for k,v in imp.items() if v>=thr]
    imp_cols = {f"importance_{k}": imp.get(k,0.0) for k in imp}

    # 2단계 CMA-ES 최적화
    bounds = {
        "vix_buy_th":(0,100),"vix_sell_th":(0,100),
        "rsi_buy_th":(0,100),"boll_buffer":(0,0.1),
        "tw_price_th":(0,20),"tm_price_th":(0,50),
        "tw_vol_th":(0,100),"tm_vol_th":(0,300),
        "rsi_sell_th":(0,100)
    }
    narrow = {}
    for k in imp_k:
        lo,hi = bounds[k]
        bp    = best_tpe[k]
        d     = 0.2*(hi-lo)
        narrow[k] = (max(lo,bp-d), min(hi,bp+d))

    def obj_cma(trial):
        p = {}
        for k,(lo,hi) in bounds.items():
            if k in imp_k:
                lo2,hi2 = narrow[k]
                if k=="vix_sell_th":
                    hi2 = min(hi2, best_tpe["vix_buy_th"])
                p[k] = trial.suggest_float(k, lo2, hi2)
            else:
                p[k] = best_tpe[k]
        return backtest_roi(p)

    cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    cma.optimize(obj_cma, n_trials=500)
    best_cma = cma.best_params

    # 최종 기록
    full_best = {**best_tpe, **best_cma}
    final_roi = cma.best_value

    rec = {
        "종목": company,
        "Start": START_DATE,
        "End": END_DATE,
        "ROI(%)": round(final_roi,2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **full_best,
        **imp_cols
    }
    new_records.append(rec)
    print(f"✅ Optimized: {company} → ROI {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 3) parameters.xlsx 에 신규 레코드만 append 및 인덱스 추가
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])

updated.insert(0, 'Index', range(1, len(updated) + 1))
updated.to_excel(PARAMETERS_PATH, index=False)
print(f"\n🏁 Parameters updated → {PARAMETERS_PATH}")


In [ ]:
from IPython.display import display
display(pd.DataFrame(new_records))

# 만약 new_records 리스트를 DataFrame으로 바로 저장하고 싶다면
df = pd.DataFrame(new_records)

# 원하는 경로와 파일명 지정
output_path = r"D:\주식\Results\Parameters\current_opt_results.xlsx"

# 엑셀로 저장 (index=False: 행 번호는 저장하지 않음)
df.to_excel(output_path, index=False)

print(f"✅ Saved optimization results to {output_path}")

# Crypto를 위한 코드 (VIX도 그냥 parameter)
# 코드 2

# is_rule_buy/sell 함수 안에 VIX 조건 포함 (매크로와 기술적 신호를 하나로 통합) 

# 순수 룰(signal-only) 백테스트

In [ ]:
# optimize_all.py

import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정 (코드 1 과 동일)
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
MACRO_FOLDER      = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 자동 추출 & 선택
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip()
if sel.lower() == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간을 사용자 입력으로 받기
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일을 YYYY-MM-DD 형식으로 입력하세요 (기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일을 YYYY-MM-DD 형식으로 입력하세요 (기본 {default_end}): ").strip()
END_DATE = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} 부터 {END_DATE} 까지\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 각 종목에 대해 최적화 수행
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    matches = glob.glob(pattern)
    if not matches:
        print(f"⚠️ 파일 없음: {pattern}")
        continue
    file_path = matches[0]
    print(f"\n🔍 Optimizing {company} ({os.path.basename(file_path)})")

    # 데이터 로드 & 필터
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)

    # VIX 병합 (와일드카드 + 누락 시 NaN 처리)
    vix_pattern = os.path.join(MACRO_FOLDER, "*VIX*.csv")
    vix_files    = glob.glob(vix_pattern)
    if not vix_files:
        print(f"⚠️ 경고: '{MACRO_FOLDER}' 폴더에 VIX 파일이 없습니다. VIX 컬럼을 NaN으로 채웁니다.")
        df['VIX'] = np.nan
    else:
        vix_path = vix_files[0]
        vix = pd.read_csv(
            vix_path,
            parse_dates=[0],
            encoding='utf-8-sig',
            names=['날짜','VIX'],
            header=0
        )
        df = df.merge(vix, on='날짜', how='left')

    # ─────────────────────────────────────────────────────────
    # 신호 함수: 이 부분만 코드 2 그대로 유지!
    # ─────────────────────────────────────────────────────────
    def is_rule_buy(r, p):
        return (
            not np.isnan(r['VIX']) and r['VIX'] >= p['vix_buy_th'] and
            r['RSI (14일)']           < p['rsi_buy_th'] and
            r['종가']                 < r['볼린저밴드 하단'] * (1 + p['boll_buffer']) and
            r['MACD']                 > r['MACD 시그널'] and
            r['SMA 5일']              > r['SMA 10일'] and
            r['SMA 5일']              < r['SMA 60일'] and
            r['가격 상승률 (2주)']     < p['tw_price_th'] and
            r['가격 상승률 (3개월)']   < p['tm_price_th'] and
            r['거래량 상승률 (2주)']   < p['tw_vol_th'] and
            r['거래량 상승률 (3개월)'] < p['tm_vol_th']
        )

    def is_rule_sell(r, p):
        return (
            not np.isnan(r['VIX']) and r['VIX'] <= p['vix_sell_th'] and
            r['RSI (14일)'] > p['rsi_sell_th']
        )

    # ─────────────────────────────────────────────────────────
    # ROI 계산 함수: signal-only 백테스트 (코드 2 그대로)
    # ─────────────────────────────────────────────────────────
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_rule_buy(r, p):
                shares, cash = cash / price, 0.0
            elif shares > 0 and is_rule_sell(r, p):
                cash, shares = shares * price, 0.0
        final = cash + shares * df.iloc[-1]['종가']
        return (final - 10_000.0) / 10_000.0 * 100

    # ─────────────────────────────────────────────────────────
    # 1단계: TPE 탐색 (코드 1 과 동일 n_trials=1500)
    # ─────────────────────────────────────────────────────────
    def obj_tpe(trial):
        return backtest_roi({
            'vix_buy_th':  trial.suggest_float("vix_buy_th",  0, 100),
            'vix_sell_th': trial.suggest_float("vix_sell_th", 0, 100),
            'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
            'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
            'tw_price_th': trial.suggest_float("tw_price_th", 0, 20),
            'tm_price_th': trial.suggest_float("tm_price_th", 0, 50),
            'tw_vol_th':   trial.suggest_float("tw_vol_th",   0, 100),
            'tm_vol_th':   trial.suggest_float("tm_vol_th",   0, 300),
            'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
        })

    tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    tpe.optimize(obj_tpe, n_trials=1500)
    best_tpe = tpe.best_params

    # ─────────────────────────────────────────────────────────
    # 중요도 계산 (코드 1 과 동일)
    # ─────────────────────────────────────────────────────────
    imp   = get_param_importances(tpe)
    thr   = 0.05
    imp_k = [k for k, v in imp.items() if v >= thr]
    imp_cols = {f"importance_{k}": imp.get(k, 0.0) for k in imp}

    # ─────────────────────────────────────────────────────────
    # 2단계: CMA-ES (코드 1 과 동일, n_trials=500 + vix_sell 제약)
    # ─────────────────────────────────────────────────────────
    bounds = {
        'vix_buy_th':  (0, 100), 'vix_sell_th': (0, 100),
        'rsi_buy_th':  (0, 100), 'boll_buffer': (0, 0.1),
        'tw_price_th': (0, 20),  'tm_price_th': (0, 50),
        'tw_vol_th':   (0, 100),'tm_vol_th':   (0, 300),
        'rsi_sell_th': (0, 100)
    }
    narrow = {}
    for k in imp_k:
        lo, hi = bounds[k]
        bp     = best_tpe[k]
        d      = 0.2 * (hi - lo)
        narrow[k] = (max(lo, bp - d), min(hi, bp + d))

    def obj_cma(trial):
        p = {}
        for k, (lo, hi) in bounds.items():
            if k in imp_k:
                lo2, hi2 = narrow[k]
                # vix_sell_th 은 반드시 vix_buy_th 이하로
                if k == "vix_sell_th":
                    hi2 = min(hi2, best_tpe["vix_buy_th"])
                p[k] = trial.suggest_float(k, lo2, hi2)
            else:
                p[k] = best_tpe[k]
        return backtest_roi(p)

    cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    cma.optimize(obj_cma, n_trials=500)
    best_cma = cma.best_params

    full_best = {**best_tpe, **best_cma}
    final_roi = cma.best_value

    # ─────────────────────────────────────────────────────────
    # 4) 결과 기록
    # ─────────────────────────────────────────────────────────
    rec = {
        "종목":       company,
        "Start":      START_DATE,
        "End":        END_DATE,
        "ROI(%)":     round(final_roi, 2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **full_best,
        **imp_cols
    }
    new_records.append(rec)
    print(f"✅ Optimized: {company} → ROI {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 5) parameters.xlsx 에 append & 1-based Index (코드 1 과 동일)
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated) + 1))
updated.to_excel(PARAMETERS_PATH, index=False)
print(f"\n🏁 Parameters updated → {PARAMETERS_PATH}")


In [ ]:
from IPython.display import display
display(pd.DataFrame(new_records))

# 만약 new_records 리스트를 DataFrame으로 바로 저장하고 싶다면
df = pd.DataFrame(new_records)

# 원하는 경로와 파일명 지정
output_path = r"D:\주식\Results\Parameters\current_opt_results.xlsx"

# 엑셀로 저장 (index=False: 행 번호는 저장하지 않음)
df.to_excel(output_path, index=False)

print(f"✅ Saved optimization results to {output_path}")

# 코드1. 시뮬레이션 (4가지 ) -주식

In [ ]:
# simulate_all.py

import os
import glob
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
MACRO_FOLDER     = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
RESULTS_ROOT     = r"C:\Users\LabPC\OneDrive\주식\Results"
PARAM_FILE       = os.path.join(RESULTS_ROOT, "Parameters", "parameters.xlsx")

os.makedirs(RESULTS_ROOT, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 1) parameters.xlsx 로드 후 사용자 선택
# ─────────────────────────────────────────────────────────────
dfp = pd.read_excel(PARAM_FILE)

print("🔔 시뮬레이션 가능 인덱스 목록:")
for _, row in dfp.iterrows():
    idx, comp, s, e, roi = int(row['Index']), row['종목'], row['Start'], row['End'], row['ROI(%)']
    print(f"  {idx}. {comp} ({s} ~ {e}, ROI: {roi:.2f}%)")

sel = input("\n시뮬레이션할 Index 번호(콤마로 구분) 또는 'all' 입력: ").strip()
if sel.lower() == 'all':
    selected = dfp.copy()
else:
    nums = [int(x) for x in sel.split(',') if x.strip().isdigit()]
    selected = dfp[dfp['Index'].isin(nums)].copy()

# ▶️ 날짜 수동 입력
custom_start, custom_end = [], []
for _, row in selected.iterrows():
    comp = row['종목']
    print(f"\n📌 종목: {comp}")
    s = input("  시작일 입력 (예: 2020-01-01) → ").strip()
    e = input("  종료일 입력 (예: 2024-12-31) → ").strip()
    custom_start.append(pd.to_datetime(s))
    custom_end.append(pd.to_datetime(e))

selected['Start'] = custom_start
selected['End']   = custom_end

print("\n▶ 선택 및 사용자 지정 날짜:")
print(selected[['Index','종목','Start','End','ROI(%)']].to_string(index=False))
print()

# ─────────────────────────────────────────────────────────────
# 신호 함수
# ─────────────────────────────────────────────────────────────
def is_macro_buy(r, p):
    return not np.isnan(r['VIX']) and r['VIX'] >= p['vix_buy_th']

def is_macro_sell(r, p):
    return not np.isnan(r['VIX']) and r['VIX'] <= p['vix_sell_th']

def is_rule_buy(r, p):
    return (
        r['RSI (14일)']           < p['rsi_buy_th'] and
        r['종가']                 < r['볼린저밴드 하단'] * (1 + p['boll_buffer']) and
        r['MACD']                 > r['MACD 시그널'] and
        r['SMA 5일']              > r['SMA 10일'] and
        r['SMA 5일']              < r['SMA 60일'] and
        r['가격 상승률 (2주)']     < p['tw_price_th'] and
        r['가격 상승률 (3개월)']   < p['tm_price_th'] and
        r['거래량 상승률 (2주)']   < p['tw_vol_th'] and
        r['거래량 상승률 (3개월)'] < p['tm_vol_th']
    )

def is_rule_sell(r, p):
    return r['RSI (14일)'] > p['rsi_sell_th']

# ─────────────────────────────────────────────────────────────
# 백테스트 & 로그 함수 (쿨다운=0 고정)
# ─────────────────────────────────────────────────────────────
def run_backtest(df, params, extra_on_buy=False, cooldown_days=0):
    cash, shares = 10_000.0, 0.0
    total_injected = 10_000.0
    logs = []
    last_buy_date = None

    for _, row in df.iterrows():
        date, price, vix = row['날짜'], row['종가'], row['VIX']
        ok_to_buy = last_buy_date is None or (date - last_buy_date).days >= cooldown_days

        # Macro Buy
        if is_macro_buy(row, params) and ok_to_buy:
            if extra_on_buy:
                cash += 10_000.0
                total_injected += 10_000.0
            shares += cash / price
            cash = 0.0
            last_buy_date = date
            logs.append([date,
                         f"BUY_MACRO_VIX (VIX={vix:.2f}≥{params['vix_buy_th']:.2f})",
                         price, shares, cash, cash + shares * price, total_injected])

        # Macro Sell
        elif shares > 0 and is_macro_sell(row, params):
            cash += shares * price
            shares = 0.0
            last_buy_date = None
            logs.append([date,
                         f"SELL_MACRO_VIX (VIX={vix:.2f}≤{params['vix_sell_th']:.2f})",
                         price, shares, cash, cash, total_injected])

        # Rule Buy
        elif is_rule_buy(row, params) and ok_to_buy:
            if extra_on_buy:
                cash += 10_000.0
                total_injected += 10_000.0
            shares += cash / price
            cash = 0.0
            last_buy_date = date
            logs.append([date,
                         f"BUY_RULE (RSI<{params['rsi_buy_th']:.2f}, ...)",
                         price, shares, cash, cash + shares * price, total_injected])

        # Rule Sell
        elif shares > 0 and is_rule_sell(row, params):
            cash += shares * price
            shares = 0.0
            last_buy_date = None
            logs.append([date,
                         f"SELL_RULE (RSI={row['RSI (14일)']:.2f}≥{params['rsi_sell_th']:.2f})",
                         price, shares, cash, cash, total_injected])

    # Liquidate
    if shares > 0:
        date, price = df.iloc[-1]['날짜'], df.iloc[-1]['종가']
        cash += shares * price
        shares = 0.0
        logs.append([date, "LIQUIDATE", price, shares, cash, cash, total_injected])

    cols = ["날짜","액션","가격","보유주","현금","총자산","투입금액"]
    df_logs = pd.DataFrame(logs, columns=cols)
    df_logs["ROI(%)"] = (df_logs["총자산"] / df_logs["투입금액"] * 100).round(2).map(lambda x: f"{x:.2f}%")
    df_logs[["현금","총자산","투입금액"]] = df_logs[["현금","총자산","투입금액"]].astype(float).applymap(lambda x: f"{x:,.0f}")
    return df_logs

# ─────────────────────────────────────────────────────────────
# 2) 선택된 레코드별 시뮬레이션 및 베이스라인 추가
# ─────────────────────────────────────────────────────────────
for _, row in selected.iterrows():
    idx       = int(row['Index'])
    comp      = row['종목']
    start, end= row['Start'], row['End']
    params    = row.drop(['Index','종목','Start','End','ROI(%)','OptimizedAt']).to_dict()

    # 지표 포함 CSV 로드 & 날짜 필터링 (종료일 포함)
    files = glob.glob(os.path.join(PROCESSED_FOLDER, f"{comp}_*_지표포함.csv"))
    df_raw = pd.read_csv(files[0], encoding='utf-8-sig')
    df_raw['날짜'] = pd.to_datetime(df_raw['날짜'])
    df = df_raw[(df_raw['날짜'] >= start) & (df_raw['날짜'] <= end)].reset_index(drop=True)

    # VIX 병합 (와일드카드)
    vix_files = glob.glob(os.path.join(MACRO_FOLDER, "*VIX*.csv"))
    if not vix_files:
        df['VIX'] = np.nan
    else:
        vix = pd.read_csv(vix_files[0], parse_dates=[0], encoding='utf-8-sig', header=0)
        vix.columns = ['날짜','VIX']
        df = df.merge(vix, on='날짜', how='left')

    out_dir = os.path.join(RESULTS_ROOT, comp)
    os.makedirs(out_dir, exist_ok=True)

    # ① 한 번만 투자
    df_once = run_backtest(df, params, extra_on_buy=False, cooldown_days=0)
    roi_once = float(df_once["ROI(%)"].iloc[-1].rstrip('%'))
    fname1 = f"{idx}_{comp}_once_{start.date()}_{end.date()}_ROI_{roi_once:.2f}.csv"
    df_once.to_csv(os.path.join(out_dir, fname1), index=False, encoding='utf-8-sig')

    # ② 매수마다 추가 투자
    df_extra = run_backtest(df, params, extra_on_buy=True, cooldown_days=0)
    roi_extra = float(df_extra["ROI(%)"].iloc[-1].rstrip('%'))
    fname2 = f"{idx}_{comp}_extra_{start.date()}_{end.date()}_ROI_{roi_extra:.2f}.csv"
    df_extra.to_csv(os.path.join(out_dir, fname2), index=False, encoding='utf-8-sig')

    # ③ Baseline 1
    min_p, max_p = df['종가'].min(), df['종가'].max()
    roi_base1 = (max_p/min_p - 1)*100
    df_base1 = pd.DataFrame([
        [df.loc[df['종가'].idxmin(), '날짜'], f"BUY at {min_p:.2f}", min_p, 10_000/min_p, pd.NA, 10_000, f"{roi_base1:.2f}%"],
        [df.loc[df['종가'].idxmax(), '날짜'], f"SELL at {max_p:.2f}", max_p, 0.0, (10_000/min_p)*max_p, (10_000/min_p)*max_p, f"{roi_base1:.2f}%"]
    ], columns=["날짜","액션","가격","보유주","현금","총자산","ROI(%)"])
    fname3 = f"{idx}_{comp}_가장저점매수_고점매도_{start.date()}_{end.date()}_ROI_{roi_base1:.2f}.csv"
    df_base1.to_csv(os.path.join(out_dir, fname3), index=False, encoding='utf-8-sig')

    # ④ Baseline 2: 일별 DCA
    days = (end-start).days
    total_inj = days*10_000.0
    daily_amt = total_inj/days
    logs, shares = [], 0.0
    for _, r in df.iterrows():
        date, price = r['날짜'], r['종가']
        shares += daily_amt/price
        logs.append([date, "DCA_BUY", price, shares, daily_amt, total_inj])
    df_dca = pd.DataFrame(logs, columns=["날짜","액션","가격","보유주","투입금액","총투입"])
    df_dca["총자산_num"] = df_dca["보유주"]*df_dca["가격"]
    df_dca["ROI_num"]    = df_dca["총자산_num"]/df_dca["총투입"]*100
    roi_base2 = df_dca["ROI_num"].iloc[-1]
    df_dca["총자산"]   = df_dca["총자산_num"].map(lambda x: f"{x:,.0f}")
    df_dca["ROI(%)"]   = df_dca["ROI_num"].round(2).map(lambda x: f"{x:.2f}%")
    df_dca["투입금액"] = df_dca["투입금액"].map(lambda x: f"{x:,.0f}")
    df_dca["총투입"]   = df_dca["총투입"].map(lambda x: f"{x:,.0f}")
    df_dca = df_dca[["날짜","액션","가격","보유주","투입금액","총자산","총투입","ROI(%)"]]
    fname4 = f"{idx}_{comp}_분할매수시나리오_{start.date()}_{end.date()}_ROI_{roi_base2:.2f}.csv"
    df_dca.to_csv(os.path.join(out_dir, fname4), index=False, encoding='utf-8-sig')

    print(f"✅ {comp} 시뮬레이션 완료: {fname1}, {fname2}, {fname3}, {fname4}")


# 코드2. Simlulation - Crypto (VIX parameterized)

In [ ]:
# simulate_all.py

import os
import glob
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# 경로 설정 (코드 1 과 동일)
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
MACRO_FOLDER     = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
RESULTS_ROOT     = r"C:\Users\LabPC\OneDrive\주식\Results"
PARAM_FILE       = os.path.join(RESULTS_ROOT, "Parameters", "parameters.xlsx")

os.makedirs(RESULTS_ROOT, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 1) parameters.xlsx 로드 후 사용자 선택
# ─────────────────────────────────────────────────────────────
dfp = pd.read_excel(PARAM_FILE)

print("🔔 시뮬레이션 가능 인덱스 목록:")
for _, row in dfp.iterrows():
    idx, comp, s, e, roi = int(row['Index']), row['종목'], row['Start'], row['End'], row['ROI(%)']
    print(f"  {idx}. {comp} ({s} ~ {e}, ROI: {roi:.2f}%)")

sel = input("\n시뮬레이션할 Index 번호(콤마로 구분) 또는 'all' 입력: ").strip()
if sel.lower() == 'all':
    selected = dfp.copy()
else:
    nums = [int(x) for x in sel.split(',') if x.strip().isdigit()]
    selected = dfp[dfp['Index'].isin(nums)].copy()

# ▶️ 날짜 수동 입력
custom_start, custom_end = [], []
for _, row in selected.iterrows():
    comp = row['종목']
    print(f"\n📌 종목: {comp}")
    s = input("  시작일 입력 (예: 2020-01-01) → ").strip()
    e = input("  종료일 입력 (예: 2024-12-31) → ").strip()
    custom_start.append(pd.to_datetime(s))
    custom_end.append(pd.to_datetime(e))

selected['Start'] = custom_start
selected['End']   = custom_end

print("\n▶ 선택 및 사용자 지정 날짜:")
print(selected[['Index','종목','Start','End','ROI(%)']].to_string(index=False))
print()

# ─────────────────────────────────────────────────────────────
# 신호 함수 (코드 2 매수/매도 로직 그대로)
# ─────────────────────────────────────────────────────────────
def is_rule_buy(r, p):
    return (
        not np.isnan(r['VIX']) and r['VIX'] >= p['vix_buy_th'] and
        r['RSI (14일)']           < p['rsi_buy_th'] and
        r['종가']                 < r['볼린저밴드 하단'] * (1 + p['boll_buffer']) and
        r['MACD']                 > r['MACD 시그널'] and
        r['SMA 5일']              > r['SMA 10일'] and
        r['SMA 5일']              < r['SMA 60일'] and
        r['가격 상승률 (2주)']     < p['tw_price_th'] and
        r['가격 상승률 (3개월)']   < p['tm_price_th'] and
        r['거래량 상승률 (2주)']   < p['tw_vol_th'] and
        r['거래량 상승률 (3개월)'] < p['tm_vol_th']
    )

def is_rule_sell(r, p):
    return (
        not np.isnan(r['VIX']) and r['VIX'] <= p['vix_sell_th'] and
        r['RSI (14일)'] > p['rsi_sell_th']
    )

# ─────────────────────────────────────────────────────────────
# 백테스트 함수 (code2 로직 그대로) + 빈 로그 처리
# ─────────────────────────────────────────────────────────────
def run_backtest(df, params, extra_on_buy=False, cooldown_days=0):
    cash, shares = 10_000.0, 0.0
    total_injected = 10_000.0
    logs, last_buy_date = [], None

    for _, r in df.iterrows():
        date, price = r['날짜'], r['종가']
        ok = last_buy_date is None or (date - last_buy_date).days >= cooldown_days

        if is_rule_buy(r, params) and ok:
            if extra_on_buy:
                cash += 10_000.0
                total_injected += 10_000.0
            shares += cash / price
            cash = 0.0
            last_buy_date = date
            logs.append([date,
                         f"BUY_RULE (VIX≥{params['vix_buy_th']:.2f})",
                         price, shares, cash, cash + shares * price, total_injected])

        elif shares > 0 and is_rule_sell(r, params):
            cash += shares * price
            shares = 0.0
            last_buy_date = None
            logs.append([date,
                         f"SELL_RULE (VIX≤{params['vix_sell_th']:.2f})",
                         price, shares, cash, cash, total_injected])

    if shares > 0:
        date, price = df.iloc[-1]['날짜'], df.iloc[-1]['종가']
        cash += shares * price
        logs.append([date, "LIQUIDATE", price, 0.0, cash, cash, total_injected])

    cols = ["날짜","액션","가격","보유주","현금","총자산","투입금액"]
    out = pd.DataFrame(logs, columns=cols)
    if out.empty:
        return out

    out["ROI(%)"] = (out["총자산"] / out["투입금액"] * 100).round(2).map(lambda x: f"{x:.2f}%")
    for c in ["현금","총자산","투입금액"]:
        out[c] = out[c].astype(float).map(lambda x: f"{x:,.0f}")
    return out

# ─────────────────────────────────────────────────────────────
# 2) 시뮬레이션 & 베이스라인 (code1 방식)
# ─────────────────────────────────────────────────────────────
for _, row in selected.iterrows():
    idx, comp = int(row['Index']), row['종목']
    start, end = row['Start'], row['End']
    params = row.drop(['Index','종목','Start','End','ROI(%)','OptimizedAt']).to_dict()

    # 지표 포함 CSV 로드 & 날짜 필터 (종료일 포함)
    fp = glob.glob(os.path.join(PROCESSED_FOLDER, f"{comp}_*_지표포함.csv"))[0]
    df_raw = pd.read_csv(fp, encoding='utf-8-sig')
    df_raw['날짜'] = pd.to_datetime(df_raw['날짜'])
    df = df_raw[(df_raw['날짜'] >= start) & (df_raw['날짜'] <= end)].reset_index(drop=True)

    # VIX 병합 (와일드카드 + NaN)
    vix_files = glob.glob(os.path.join(MACRO_FOLDER, "*VIX*.csv"))
    if not vix_files:
        df['VIX'] = np.nan
    else:
        v = pd.read_csv(vix_files[0], parse_dates=[0], encoding='utf-8-sig', header=0)
        v.columns = ['날짜','VIX']
        df = df.merge(v, on='날짜', how='left')

    out_dir = os.path.join(RESULTS_ROOT, comp)
    os.makedirs(out_dir, exist_ok=True)

    # ① 한 번만 투자
    df_once = run_backtest(df, params, extra_on_buy=False, cooldown_days=0)
    if not df_once.empty:
        roi_once = float(df_once["ROI(%)"].iloc[-1].rstrip('%'))
        fname1 = f"{idx}_{comp}_once_{start.date()}_{end.date()}_ROI_{roi_once:.2f}.csv"
        df_once.to_csv(os.path.join(out_dir, fname1), index=False, encoding='utf-8-sig')

    # ② 매수마다 추가 투자
    df_extra = run_backtest(df, params, extra_on_buy=True, cooldown_days=0)
    if not df_extra.empty:
        roi_extra = float(df_extra["ROI(%)"].iloc[-1].rstrip('%'))
        fname2 = f"{idx}_{comp}_extra_{start.date()}_{end.date()}_ROI_{roi_extra:.2f}.csv"
        df_extra.to_csv(os.path.join(out_dir, fname2), index=False, encoding='utf-8-sig')

    # ③ Baseline 1: 전체 기간 최저가 → 최고가
    min_p, max_p = df['종가'].min(), df['종가'].max()
    roi_base1 = (max_p / min_p - 1) * 100
    df_base1 = pd.DataFrame([
        [df.loc[df['종가'].idxmin(), '날짜'], f"BUY at {min_p:.2f}", min_p, 10_000/min_p, pd.NA, 10_000, f"{roi_base1:.2f}%"],
        [df.loc[df['종가'].idxmax(), '날짜'], f"SELL at {max_p:.2f}", max_p, 0.0, (10_000/min_p)*max_p, (10_000/min_p)*max_p, f"{roi_base1:.2f}%"]
    ], columns=["날짜","액션","가격","보유주","현금","총자산","ROI(%)"])
    fname3 = f"{idx}_{comp}_가장저점매수_고점매도_{start.date()}_{end.date()}_ROI_{roi_base1:.2f}.csv"
    df_base1.to_csv(os.path.join(out_dir, fname3), index=False, encoding='utf-8-sig')

    # ④ Baseline 2: 일별 DCA
    days = (end - start).days
    total_inj = days * 10_000.0
    daily_amt = total_inj / days
    logs, shares = [], 0.0
    for _, r in df.iterrows():
        date, price = r['날짜'], r['종가']
        shares += daily_amt / price
        logs.append([date, "DCA_BUY", price, shares, daily_amt, total_inj])
    df_dca = pd.DataFrame(logs, columns=["날짜","액션","가격","보유주","투입금액","총투입"])
    df_dca["총자산_num"] = df_dca["보유주"] * df_dca["가격"]
    df_dca["ROI_num"]    = df_dca["총자산_num"] / df_dca["총투입"] * 100
    roi_base2 = df_dca["ROI_num"].iloc[-1]
    df_dca["총자산"]   = df_dca["총자산_num"].map(lambda x: f"{x:,.0f}")
    df_dca["ROI(%)"]   = df_dca["ROI_num"].round(2).map(lambda x: f"{x:.2f}%")
    df_dca["투입금액"] = df_dca["투입금액"].map(lambda x: f"{x:,.0f}")
    df_dca["총투입"]   = df_dca["총투입"].map(lambda x: f"{x:,.0f}")
    df_dca = df_dca[["날짜","액션","가격","보유주","투입금액","총자산","총투입","ROI(%)"]]
    fname4 = f"{idx}_{comp}_분할매수시나리오_{start.date()}_{end.date()}_ROI_{roi_base2:.2f}.csv"
    df_dca.to_csv(os.path.join(out_dir, fname4), index=False, encoding='utf-8-sig')

    print(f"✅ {comp} 시뮬레이션 완료: "
          f"{fname1 if not df_once.empty else '(no trades)'}, "
          f"{fname2 if not df_extra.empty else '(no trades)'}, "
          f"{fname3}, {fname4}")


In [ ]:
# Visualization

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter
import os

# 1) 엑셀 파일 경로
PARAMETERS_PATH = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters\parameters.xlsx"
if not os.path.exists(PARAMETERS_PATH):
    raise FileNotFoundError(f"parameters.xlsx 파일을 찾을 수 없습니다:\n{PARAMETERS_PATH}")

# 2) 파라미터/importance 리스트 정의
param_cols = [
    "vix_buy_th","vix_sell_th","rsi_buy_th","boll_buffer",
    "tw_price_th","tm_price_th","tw_vol_th","tm_vol_th","rsi_sell_th"
]
imp_cols = [f"importance_{c}" for c in param_cols]

# 3) pandas로 읽어서 컬럼 재정렬
df = pd.read_excel(PARAMETERS_PATH)
# 나머지(other) 컬럼(인덱스, 종목, 기간, ROI, OptimizedAt 등)
other = [c for c in df.columns if c not in (param_cols + imp_cols)]
new_order = other + param_cols + imp_cols
df = df.reindex(columns=new_order)
df.to_excel(PARAMETERS_PATH, index=False)

# 4) importance 최소·최대 계산 (NaN 제외)
imp_min = df[imp_cols].min(skipna=True)
imp_max = df[imp_cols].max(skipna=True)

# 5) threshold 설정 (0.02 미만은 무시)
THRESHOLD = 0.02

# 6) openpyxl로 동일 파일 열기
wb = openpyxl.load_workbook(PARAMETERS_PATH)
ws = wb.active

# 7) 헤더(1행)에서 컬럼 이름→열 번호 매핑
header_idx = {cell.value: cell.column for cell in ws[1]}

# 8) 각 행(row)마다 파라미터 컬럼에 importance 기반 색칠
for row_idx in range(2, len(df) + 2):
    for param, imp in zip(param_cols, imp_cols):
        importance = df.at[row_idx-2, imp]
        if pd.isna(importance) or importance < THRESHOLD:
            continue
        lo, hi = imp_min[imp], imp_max[imp]
        ratio = (importance - lo) / (hi - lo) if hi > lo else 0.0
        r = int(255 * (1 - ratio))
        g = int(255 * ratio)
        fill = PatternFill(
            start_color=f"{r:02X}{g:02X}00",
            end_color=f"{r:02X}{g:02X}00",
            fill_type="solid"
        )
        col_letter = get_column_letter(header_idx[param])
        ws[f"{col_letter}{row_idx}"].fill = fill

# 9) 덮어쓰기
wb.save(PARAMETERS_PATH)
print("✅ parameters.xlsx 컬럼 재배치 및 importance 기반 색칠 완료")